In [1]:
from gensim.models import Word2Vec
from konlpy.tag import Komoran
import time
import pandas as pd

In [3]:
df = pd.read_csv("./data/nsmc/ratings.txt", sep = "\t")

In [4]:
df = df.dropna()

In [5]:
df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [6]:
komoran = Komoran()

In [7]:
df.shape

(199992, 3)

In [8]:
%%time
# 문장 단위로 명사만 추출해 학습 입력 데이터로 만듦
docs = df["document"].map(lambda x: komoran.nouns(x))

CPU times: total: 1min 56s
Wall time: 1min 38s


In [9]:
docs

0                                                       [때]
1         [디자인, 학생, 외국, 디자이너, 전통, 발전, 문화, 산업, 사실, 우리나라, ...
2                                    [폴리스, 스토리, 시리즈, 뉴, 최고]
3                                [연기, 것, 라고, 생각, 몰입, 영, 화지]
4                                        [안개, 밤하늘, 초승달, 영화]
                                ...                        
199995                                         [포켓, 몬스터, 짜]
199996                                                   []
199997                            [완전, 사이코, 영화, 마지막, 영화, 질]
199998                                          [라따뚜이, 스머프]
199999                       [포, 풍, 저그, 가나, 신다영, 차영, 차영, 차]
Name: document, Length: 199992, dtype: object

In [10]:
# word2vec 모델 학습
model = Word2Vec(sentences = docs, vector_size = 256, window = 4, min_count = 2, sg = 1)

- word2vec 의 주요 하이퍼파라미터
    - sentences : Word2Vec 모델 학습에 필요한 문장 데이터
    - vector_size : 단어 임베딩 벡터의 차원(크기)
    - window : 주변 단어 윈도우의 크기
    - min_count : 단어 최소 빈도 수 제한(설정된 min_count 빈도 수 미만의 단어들은 학습하지 않음)
    - sg : 0(CBOW 모델), 1(skip-gram 모델)

In [11]:
# 모델 저장
model.save("./model/nsmc_w2v.model")

In [12]:
# 학습된 말뭉치 개수, 코퍼스 내 전체 단어 개수
print("corpus_count :", model.corpus_count)
print("corpus_total_words :", model.corpus_total_words)

corpus_count : 199992
corpus_total_words : 1076840


# 모델 활용

In [13]:
# 모델 불러오기
model = Word2Vec.load("./model/nsmc_w2v.model")

In [14]:
model.corpus_total_words

1076840

In [15]:
# "사랑" 이란 단어로 생성한 단어 임베딩 벡터
print(model.wv["사랑"])

[ 0.01864792 -0.13357377 -0.25197557 -0.06051022  0.17742743  0.06030699
 -0.01835854 -0.01319071 -0.23166972  0.24743107 -0.32820246 -0.06927653
 -0.02590778  0.3274374  -0.05835712  0.30050302 -0.23585358 -0.10666603
 -0.10337335  0.3674452  -0.14806765 -0.16323063 -0.10760664 -0.4169987
 -0.18135689  0.0878128   0.15534069  0.04223175 -0.3824319  -0.36582848
  0.21525136  0.09592939 -0.07821866  0.218168   -0.29746443  0.27297804
 -0.12683304  0.01024595  0.013106   -0.18553406 -0.36856505  0.2791195
 -0.13628154 -0.05431008  0.45001802  0.40691587 -0.01503368  0.18777835
  0.17477037 -0.25637698 -0.2771073  -0.02607157  0.17826232  0.3847636
  0.00218974  0.3261955   0.03016613 -0.14032893  0.10724586  0.32526833
  0.03972282 -0.1025584   0.31257445  0.18009946 -0.06820546  0.08344957
  0.00253413  0.20438555 -0.25218698  0.1273167   0.3205914   0.17804968
  0.18045537 -0.11815605  0.19969645 -0.2923668  -0.08045402 -0.00447769
  0.33407873  0.45115006  0.11532147 -0.1483484  -0.33

- 모델을 학습할 때 설정한 vector_size 하이퍼파라미터만큼 단어 임베딩 벡터 차원 크기가 결정됨

In [16]:
# 단어 유사도 계산
print("일요일 = 월요일", model.wv.similarity(w1 = "일요일", w2 = "월요일"))

일요일 = 월요일 0.90711236


In [17]:
print("안성기 = 배우", model.wv.similarity(w1 = "안성기", w2 = "배우"))

안성기 = 배우 0.71747124


In [18]:
print("대기업 = 삼성", model.wv.similarity(w1 = "대기업", w2 = "삼성"))

대기업 = 삼성 0.8872664


In [19]:
print("일요일 = 삼성", model.wv.similarity(w1 = "일요일", w2 = "삼성"))

일요일 = 삼성 0.64122516


In [24]:
print("히어로 = 삼성", model.wv.similarity(w1 = "히어로", w2 = "삼성"))

히어로 = 삼성 0.42529297


- model.wv.similarity() : 두 단어 간의 유사도를 계산

In [25]:
# 가장 유사한 단어 추출
print(model.wv.most_similar("안성기", topn = 5))

[('송강호', 0.9457641243934631), ('한석규', 0.9451687335968018), ('최민식', 0.9414653778076172), ('이나영', 0.9392165541648865), ('정재영', 0.9356375932693481)]


In [26]:
print(model.wv.most_similar("시리즈", topn = 5))

[('엑스맨', 0.8233566880226135), ('포터', 0.8068362474441528), ('반지의 제왕', 0.7964420914649963), ('해리', 0.7906635999679565), ('미이라', 0.7876506447792053)]


- model.wv.most_similar() 함수 : 인자로 사용한 단어와 가장 유사한 단어를 리스트로 반환
    - 벡터 공간에서 가장 가까운 거리에 있는 단어들을 반환
    - topn : 반환되는 유사한 단어 수
 
- 유사도가 1에 가까울수록 두 단어는 동일한 의미이거나 문법적으로 관련이 있을 가능성이 높음

In [27]:
import gensim

In [28]:
sentences = gensim.models.word2vec.Text8Corpus("./data/text8.txt")

In [29]:
model = gensim.models.word2vec.Word2Vec(sentences, vector_size = 256)

In [30]:
model.wv.most_similar(positive = ["king", "woman"], negative = ["man"])

[('queen', 0.6704072952270508),
 ('elizabeth', 0.5751454830169678),
 ('daughter', 0.5473021268844604),
 ('throne', 0.5460261106491089),
 ('prince', 0.5444624423980713),
 ('son', 0.5312409400939941),
 ('princess', 0.526008129119873),
 ('emperor', 0.5216237306594849),
 ('monarch', 0.521369218826294),
 ('empress', 0.5209083557128906)]